# Kuliah #10: Ketidaktentuan dan Komplementaritas

Buku catatan interaktif (*Jupyter Notebook*) ini merupakan pendamping komputasional untuk **Modul Kuliah #10: Ketidaktentuan dan Komplementaritas**. Dalam modul ini, kita menelaah dua ciri paling fundamental dari mekanika kuantum yang membatasi informasi yang dapat diperoleh dari suatu sistem fisis:

1. **Hubungan Ketidaktentuan (*Indeterminacy Relation*)**: Menyatakan batas bawah bagi hasil kali dispersi statistik (simpangan baku) hasil pengukuran dua *observable* yang tidak komutatif pada ensambel sistem yang sama.
2. **Komplementaritas (*Complementarity*)**: Menegaskan bahwa aspek-aspek fisis yang saling eksklusif (seperti sifat gelombang dan sifat partikel) dari objek kuantum tidak dapat dimanifestasikan secara serentak dalam satu pengaturan eksperimen yang sama.

Setiap konsep, persamaan, dan soal latihan dalam modul ini direkonstruksi secara lengkap menggunakan **notasi Dirac**, **mekanika matriks $2 \times 2$**, diverifikasi secara numerik dengan **Python (`numpy`)**, serta divisualisasikan dengan **`matplotlib`**.

In [1]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# Pengaturan presisi cetak matriks
np.set_printoptions(precision=4, suppress=True)
sp.init_printing()

def deg(theta_deg):
    """Konversi sudut dari derajat ke radian."""
    return np.deg2rad(theta_deg)

def clean_array(A, tol=1e-12):
    """Membersihkan nilai elemen numerik yang mendekati nol akibat floating-point error."""
    A = np.array(A, dtype=complex)
    A[np.abs(A.real) < tol] = 1j * A[np.abs(A.real) < tol].imag
    A[np.abs(A.imag) < tol] = A[np.abs(A.imag) < tol].real
    return A

def print_matrix(name, M):
    """Mencetak matriks atau vektor dengan format yang bersih."""
    M = clean_array(M)
    print(f"{name} =")
    print(M)
    print()

# ==========================================================
# DEFINISI VEKTOR BASIS POLARISASI DALAM BASIS HV
# ==========================================================
# Basis Horizontal |H> dan Vertikal |V>
P_H = np.array([[1], [0]], dtype=complex)
P_V = np.array([[0], [1]], dtype=complex)

# Basis Diagonal |+45> dan |-45>
P_plus_45 = (1 / np.sqrt(2)) * np.array([[1], [1]], dtype=complex)
P_minus_45 = (1 / np.sqrt(2)) * np.array([[1], [-1]], dtype=complex)

# Basis Melingkar Kiri |L> dan Melingkar Kanan |R>
P_L = (1 / np.sqrt(2)) * np.array([[1], [1j]], dtype=complex)
P_R = (1 / np.sqrt(2)) * np.array([[1], [-1j]], dtype=complex)

# ==========================================================
# DEFINISI OPERATOR POLARISASI DALAM BASIS HV
# ==========================================================
# Operator P_HV (ekuivalen dengan Pauli-Z)
J_HV = np.array([[1, 0], [0, -1]], dtype=complex)

# Operator P_45 (ekuivalen dengan Pauli-X)
J_45 = np.array([[0, 1], [1, 0]], dtype=complex)

# Operator P_C (ekuivalen dengan Pauli-Y)
J_C = np.array([[0, -1j], [1j, 0]], dtype=complex)

# ==========================================================
# FUNGSI UTILITAS STATISTIKA KUANTUM
# ==========================================================
def expectation_value(op, state):
    """Menghitung nilai harap <state| op |state>."""
    state = np.array(state, dtype=complex)
    return np.vdot(state, op @ state)

def variance(op, state):
    """Menghitung variansi (Delta A)^2 = <A^2> - <A>^2."""
    exp_op = expectation_value(op, state)
    op_sq = op @ op
    exp_op_sq = expectation_value(op_sq, state)
    return np.real(exp_op_sq - exp_op**2)

def std_dev(op, state):
    """Menghitung simpangan baku Delta A = sqrt(variansi)."""
    var = variance(op, state)
    return np.sqrt(np.maximum(0.0, var))

def commutator(A, B):
    """Menghitung komutator [A, B] = AB - BA."""
    return A @ B - B @ A

print("Setup selesai. Operator dasar dan fungsi statistika kuantum siap digunakan.")

# Komutator dan Hubungan Ketidaktentuan

Dalam mekanika kuantum, setiap besaran yang dapat diukur (*observable*) direpresentasikan oleh operator linear Hermitian yang bertindak pada ruang Hilbert keadaan sistem. Secara umum, dua operator $\hat{A}$ dan $\hat{B}$ tidak bersifat komutatif, yaitu $\hat{A}\hat{B} \neq \hat{B}\hat{A}$. Oleh karena itu, kita mendefinisikan **komutator** dari $\hat{A}$ dan $\hat{B}$ sebagai:

$$
[\hat{A}, \hat{B}] \equiv \hat{A}\hat{B} - \hat{B}\hat{A}, \tag{1}
$$

yang juga merupakan sebuah operator. Jika $[\hat{A}, \hat{B}] = 0$, kedua operator tersebut dikatakan **komut** (*commute*), dan urutan operasinya tidaklah penting.

Sekarang tinjau *observable* $A$ dan $B$, dengan operator Hermitian $\hat{A}$ dan $\hat{B}$ yang bersesuaian. Kita melakukan serangkaian pengukuran $A$ pada sekumpulan besar sistem yang identik, yang semuanya disiapkan dalam keadaan $|\psi\rangle$, atau biasa disebut sebagai ensambel (*ensemble*). Ketidakpastian dalam pengukuran semacam itu dinyatakan sebagai simpangan baku $\Delta A$, yang merupakan akar kuadrat dari variansi:

$$
\begin{aligned}
\Delta A^2 &= \langle (\hat{A} - \langle \hat{A} \rangle)^2 \rangle \\
&= \langle \psi | (\hat{A} - \langle \hat{A} \rangle)^\dagger (\hat{A} - \langle \hat{A} \rangle) | \psi \rangle,
\end{aligned} \tag{2}
$$

dengan baris kedua berlaku karena $\hat{A}$ dan skalar $\langle \hat{A} \rangle$ bersifat Hermitian. Jika kita mendefinisikan keadaan baru sebagai:

$$
|a\rangle \equiv (\hat{A} - \langle \hat{A} \rangle) |\psi\rangle, \tag{3}
$$

kita peroleh $\Delta A^2 = \langle a | a \rangle$. Setelah melakukan pengukuran $A$, kita ganti perangkat kita untuk melakukan serangkaian pengukuran $B$ pada ensambel baru yang disiapkan dalam keadaan $|\psi\rangle$ yang sama. Dengan mendefinisikan:

$$
|b\rangle \equiv (\hat{B} - \langle \hat{B} \rangle) |\psi\rangle, \tag{4}
$$

variansi dari pengukuran ini adalah $\Delta B^2 = \langle b | b \rangle$. **Ketidaksamaan Schwarz** (*Cauchy-Schwarz inequality*) memberikan hubungan antar-hasil kali dalam (*inner product*) sebagai berikut:

$$
\langle a | a \rangle \langle b | b \rangle \ge |\langle a | b \rangle|^2, \tag{5}
$$

yang berarti

$$
\Delta A^2 \Delta B^2 \ge |\langle a | b \rangle|^2 \implies \Delta A \Delta B \ge |\langle a | b \rangle|. \tag{6}
$$

Karena $\langle a | b \rangle$ secara umum bernilai kompleks, berlaku hubungan:

$$
|\langle a | b \rangle|^2 = [\text{Re}(\langle a | b \rangle)]^2 + [\text{Im}(\langle a | b \rangle)]^2 \ge [\text{Im}(\langle a | b \rangle)]^2 = \left[ \frac{1}{2i} (\langle a | b \rangle - \langle b | a \rangle) \right]^2. \tag{7}
$$

Lebih lanjut, kita dapat menguraikan:

$$
\begin{aligned}
\langle a | b \rangle &= \langle \psi | (\hat{A} - \langle \hat{A} \rangle)^\dagger (\hat{B} - \langle \hat{B} \rangle) | \psi \rangle \\
&= \langle (\hat{A} - \langle \hat{A} \rangle)(\hat{B} - \langle \hat{B} \rangle) \rangle \\
&= \langle \hat{A}\hat{B} - \hat{A}\langle \hat{B} \rangle - \hat{B}\langle \hat{A} \rangle + \langle \hat{A} \rangle \langle \hat{B} \rangle \rangle \\
&= \langle \hat{A}\hat{B} \rangle - \langle \hat{A} \rangle \langle \hat{B} \rangle.
\end{aligned} \tag{8}
$$

Persamaan untuk $\langle b | a \rangle$ akan terlihat sama, tetapi dengan posisi $A$ dan $B$ yang tertukar ($\langle b | a \rangle = \langle \hat{B}\hat{A} \rangle - \langle \hat{A} \rangle \langle \hat{B} \rangle$). Sekarang gabungkan Pers. (6), (7), dan (8) untuk menghasilkan:

$$
\begin{aligned}
\Delta A \Delta B &\ge \left| \frac{1}{2i} (\langle a | b \rangle - \langle b | a \rangle) \right| \\
&= \frac{1}{2} \left| \langle \hat{A}\hat{B} \rangle - \langle \hat{A} \rangle \langle \hat{B} \rangle - \langle \hat{B}\hat{A} \rangle + \langle \hat{A} \rangle \langle \hat{B} \rangle \right| \\
&= \frac{1}{2} |\langle \hat{A}\hat{B} - \hat{B}\hat{A} \rangle|.
\end{aligned} \tag{9}
$$

Baris terakhir ini tidak lain adalah komutator, sehingga kita memperoleh **Hubungan Ketidaktentuan Robertson-Heisenberg**:

$$
\Delta A \Delta B \ge \frac{1}{2} |\langle [\hat{A}, \hat{B}] \rangle|. \tag{10}
$$

---

## Penjelasan Konseptual dan Pembuktian Matriks $2 \times 2$

### 1. Interpretasi Fisis
Persamaan (10) menunjukkan bahwa perkalian ketidakpastian pengamatan dua *observable* dikendalikan oleh nilai harap komutatornya. Jika komutator bernilai tidak nol, maka kedua *observable* tersebut tidak dapat memiliki nilai pasti secara serentak. Ini adalah konsekuensi matematis dari struktur ruang Hilbert, bukan keterbatasan teknologi instrumen ukur.

### 2. Rekonstruksi Pembuktian Menggunakan Mekanika Matriks
Mari kita re-derivasi langkah-langkah notasi Dirac di atas dengan representasi matriks kolom dan baris $2 \times 2$. Misalkan $|\psi\rangle$ adalah vektor kolom:

$$
|\psi\rangle = \begin{bmatrix} c_1 \\ c_2 \end{bmatrix}, \qquad \langle \psi | = \begin{bmatrix} c_1^* & c_2^* \end{bmatrix}, \qquad |c_1|^2 + |c_2|^2 = 1.
$$

Operator Hermitian $\hat{A}$ dan $\hat{B}$ direpresentasikan oleh matriks $\mathbf{A}$ dan $\mathbf{B}$. Nilai harap riilnya adalah skalar $a_0 \equiv \langle \hat{A} \rangle = \langle \psi | \mathbf{A} | \psi \rangle$ dan $b_0 \equiv \langle \hat{B} \rangle = \langle \psi | \mathbf{B} | \psi \rangle$. Vektor $|a\rangle$ dan $|b\rangle$ dalam bentuk vektor kolom adalah:

$$
|a\rangle = (\mathbf{A} - a_0\mathbf{I}) \begin{bmatrix} c_1 \\ c_2 \end{bmatrix}, \qquad |b\rangle = (\mathbf{B} - b_0\mathbf{I}) \begin{bmatrix} c_1 \\ c_2 \end{bmatrix}.
$$

Ketika kita menghitung hasil kali dalam $\langle a | b \rangle$, kita mengalikan vektor baris $(|a\rangle)^\dagger$ dengan vektor kolom $|b\rangle$:

$$
\begin{aligned}
\langle a | b \rangle &= \left[ (\mathbf{A} - a_0\mathbf{I}) |\psi\rangle \right]^\dagger \left[ (\mathbf{B} - b_0\mathbf{I}) |\psi\rangle \right] \\
&= \langle \psi | (\mathbf{A}^\dagger - a_0\mathbf{I}) (\mathbf{B} - b_0\mathbf{I}) | \psi \rangle.
\end{aligned}
$$

Karena $\mathbf{A}^\dagger = \mathbf{A}$, perkalian matriks di dalam menghasilkan:

$$
(\mathbf{A} - a_0\mathbf{I})(\mathbf{B} - b_0\mathbf{I}) = \mathbf{A}\mathbf{B} - a_0\mathbf{B} - b_0\mathbf{A} + a_0 b_0 \mathbf{I}.
$$

Mengalikan dari kiri dengan $\langle \psi |$ dan dari kanan dengan $|\psi\rangle$ memberikan:

$$
\begin{aligned}
\langle a | b \rangle &= \langle \psi | \mathbf{A}\mathbf{B} | \psi \rangle - a_0 \langle \psi | \mathbf{B} | \psi \rangle - b_0 \langle \psi | \mathbf{A} | \psi \rangle + a_0 b_0 \langle \psi | \psi \rangle \\
&= \langle \hat{A}\hat{B} \rangle - a_0 b_0 - b_0 a_0 + a_0 b_0 \\
&= \langle \hat{A}\hat{B} \rangle - \langle \hat{A} \rangle \langle \hat{B} \rangle.
\end{aligned}
$$

Hal yang sama berlaku untuk $\langle b | a \rangle = \langle \hat{B}\hat{A} \rangle - \langle \hat{A} \rangle \langle \hat{B} \rangle$. Pengurangan keduanya langsung menghasilkan komutator matriks $[\mathbf{A}, \mathbf{B}] = \mathbf{A}\mathbf{B} - \mathbf{B}\mathbf{A}$, membuktikan kesetaraan sempurna antara mekanika matriks dan notasi Dirac.

In [2]:
# Verifikasi Numerik Persamaan (10) dengan Matriks Polarisasi dan Keadaan Acak
print("=== VERIFIKASI NUMERIK PERSAMAAN (10) UNTUK KEADAAN SEMBARANG ===")

# Kita gunakan operator polarisasi A = P_HV dan B = P_45
A = J_HV
B = J_45

# Buat keadaan acak ternormalisasi
psi = np.array([[0.6 + 0.3j], [0.5 - np.sqrt(0.3)*1j]], dtype=complex)
psi = psi / np.linalg.norm(psi)

print_matrix("Keadaan |psi>", psi)

# Hitung simpangan baku
std_A = std_dev(A, psi)
std_B = std_dev(B, psi)
lhs = std_A * std_B

# Hitung komutator dan sisi kanan
comm_AB = commutator(A, B)
exp_comm = expectation_value(comm_AB, psi)
rhs = 0.5 * np.abs(exp_comm)

print(f"Simpangan baku Delta A           = {std_A:.6f}")
print(f"Simpangan baku Delta B           = {std_B:.6f}")
print(f"Sisi Kiri (Delta A * Delta B)    = {lhs:.6f}")
print(f"Sisi Kanan (0.5 * |<[A, B]>|)    = {rhs:.6f}")
print(f"Apakah Delta A * Delta B >= 0.5 * |<[A, B]>| terpenuhi? {lhs >= rhs - 1e-12}\n")

=== VERIFIKASI NUMERIK PERSAMAAN (10) UNTUK KEADAAN SEMBARANG ===
Keadaan |psi> =
[[0.6+0.3j    ]
 [0.5-0.5477j]]

Simpangan baku Delta A           = 0.812404
Simpangan baku Delta B           = 0.940213
Sisi Kiri (Delta A * Delta B)    = 0.763833
Sisi Kanan (0.5 * |<[A, B]>|)    = 0.547723
Apakah Delta A * Delta B >= 0.5 * |<[A, B]>| terpenuhi? True



# Ketidaktentuan (*indeterminacy*) vs. Ketidakpastian (*uncertainty*)

Persamaan (10) adalah karakteristik fundamental dari pengamatan kuantum. Istilah filosofis yang lebih tepat untuk menggambarkan relasi ini adalah **hubungan ketidaktentuan (*indeterminacy relation*)** alih-alih **hubungan ketidakpastian (*uncertainty relation*)**.

Istilah *ketidakpastian* sering kali menyiratkan bahwa sistem sebenarnya memiliki sifat pasti, namun ketidaksempurnaan alat ukur atau gangguan aktivitas pengukurlah yang menyebabkan kesalahan pengamatan. Sebaliknya, istilah *ketidaktentuan* menegaskan bahwa sifat-sifat yang bersesuaian dengan dua *observable* yang tidak komutatif **secara inheren memang tidak terdefinisi dengan baik dalam sistem kuantum sebelum dilakukan pengukuran**. Ketidaktentuan tersebut sudah melekat pada vektor keadaan $|\psi\rangle$ itu sendiri.

Perlu ditekankan pula bahwa Persamaan (10) **berlaku untuk pengukuran $A$ dan $B$ yang dilakukan pada sub-ensambel yang terpisah**:
- Kita menyiapkan banyak foton identik dalam keadaan $|\psi\rangle$.
- Pada separuh kelompok foton, kita mengukur $A$ untuk mendapatkan $\Delta A$.
- Pada separuh kelompok foton lainnya (yang masih dalam keadaan $|\psi\rangle$ utuh), kita mengukur $B$ untuk mendapatkan $\Delta B$.

Jika kita mencoba mengukur $\hat{\mathcal{P}}_{HV}$ sekaligus $\hat{\mathcal{P}}_{45}$ berurutan pada satu foton yang sama, pengukuran pertama ($\hat{\mathcal{P}}_{HV}$) akan **meruntuhkan (*collapse*)** keadaan sistem menjadi $|H\rangle$ atau $|V\rangle$. Keadaan foton telah berubah sebelum pengukuran kedua ($\hat{\mathcal{P}}_{45}$) dilakukan, sehingga tidak lagi valid menggunakan keadaan awal $|\psi\rangle$ dalam evaluasi statistik.

---

## Ketidakpastian Pengukuran Polarisasi pada Keadaan $|R\rangle$

Seberkas foton disiapkan dalam keadaan polarisasi melingkar kanan $|R\rangle$. Dalam basis $HV$, keadaan ini dituliskan sebagai:

$$
|R\rangle = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ -i \end{pmatrix}_{HV}.
$$

Kita ingin menentukan ketidakpastian pengukuran $\hat{\mathcal{P}}_{HV}$ dan $\hat{\mathcal{P}}_{45}$ yang dilakukan secara terpisah pada berkas ini, dan menunjukkan bahwa hasilnya konsisten dengan hubungan ketidaktentuan:

$$
\Delta \mathcal{P}_{HV} \Delta \mathcal{P}_{45} \ge \frac{1}{2} |\langle [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}] \rangle|. \tag{11}
$$

### Analisis Langkah demi Langkah:
1. **Nilai Harap dan Variansi untuk $\hat{\mathcal{P}}_{HV}$**:
   Matriks operator $\hat{\mathcal{P}}_{HV} \doteq \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$. Kita hitung $\hat{\mathcal{P}}_{HV} |R\rangle$:

   $$
   \hat{\mathcal{P}}_{HV} |R\rangle = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ -i \end{pmatrix} = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ i \end{pmatrix} = |L\rangle.
   $$

   Karena $\langle R | L \rangle = 0$, nilai harapnya adalah $\langle \hat{\mathcal{P}}_{HV} \rangle = 0$. Karena $\hat{\mathcal{P}}_{HV}^2 = \hat{\mathbf{I}}$, variansinya adalah:

   $$
   \Delta \mathcal{P}_{HV}^2 = \langle \hat{\mathcal{P}}_{HV}^2 \rangle - \langle \hat{\mathcal{P}}_{HV} \rangle^2 = 1 - 0 = 1 \implies \Delta \mathcal{P}_{HV} = 1.
   $$

2. **Nilai Harap dan Variansi untuk $\hat{\mathcal{P}}_{45}$**:
   Representasi matriks $\hat{\mathcal{P}}_{45} \doteq \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$. Kita hitung nilai harapnya:

   $$
   \langle \hat{\mathcal{P}}_{45} \rangle = \langle R | \hat{\mathcal{P}}_{45} | R \rangle = \frac{1}{2} \begin{pmatrix} 1 & i \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 1 \\ -i \end{pmatrix} = \frac{1}{2} \begin{pmatrix} 1 & i \end{pmatrix} \begin{pmatrix} -i \\ 1 \end{pmatrix} = \frac{1}{2} (-i + i) = 0.
   $$

   Karena $\hat{\mathcal{P}}_{45}^2 = \hat{\mathbf{I}}$, maka $\langle \hat{\mathcal{P}}_{45}^2 \rangle = 1$. Variansinya menjadi:

   $$
   \Delta \mathcal{P}_{45}^2 = 1 - 0 = 1 \implies \Delta \mathcal{P}_{45} = 1.
   $$

3. **Evaluasi Komutator $[\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}]$**:
   Perkalian matriks komutator memberikan:

   $$
   [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}] = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} - \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} = \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix} - \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} 0 & 2 \\ -2 & 0 \end{pmatrix}.
   $$

   Nilai harap komutator pada keadaan $|R\rangle$ adalah:

   $$
   \langle [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}] \rangle = \frac{1}{2} \begin{pmatrix} 1 & i \end{pmatrix} \begin{pmatrix} 0 & 2 \\ -2 & 0 \end{pmatrix} \begin{pmatrix} 1 \\ -i \end{pmatrix} = \frac{1}{2} \begin{pmatrix} 1 & i \end{pmatrix} \begin{pmatrix} -2i \\ -2 \end{pmatrix} = \frac{1}{2} (-2i - 2i) = -2i.
   $$

4. **Verifikasi Hubungan Ketidaktentuan Persamaan (11)**:
   Substitusi ke dalam relasi:

   $$
   (1)(1) \ge \frac{1}{2} |-2i| \implies 1 \ge 1.
   $$

Keadaan $|R\rangle$ memenuhi kesamaan persis ($1=1$). Oleh karena itu, $|R\rangle$ disebut sebagai **keadaan ketidakpastian minimum (*minimum uncertainty state*)** untuk pengamatan $\hat{\mathcal{P}}_{HV}$ dan $\hat{\mathcal{P}}_{45}$.

In [3]:
# Verifikasi Numerik Ketidakpastian Polarisasi pada Keadaan |R>
print("=== VERIFIKASI KETIDAKPASTIAN POLARISASI PADA KEADAAN |R> ===")

state_R = P_R
print_matrix("P_R", state_R)

std_hv = std_dev(J_HV, state_R)
std_45 = std_dev(J_45, state_R)
lhs_val = std_hv * std_45

comm_val = expectation_value(commutator(J_HV, J_45), state_R)
rhs_val = 0.5 * np.abs(comm_val)

print(f"Simpangan baku Delta P_HV        = {std_hv:.6f}")
print(f"Simpangan baku Delta P_45        = {std_45:.6f}")
print(f"Sisi Kiri (Delta P_HV * Delta P_45) = {lhs_val:.6f}")
print(f"Nilai Harap Komutator            = {comm_val:.4f}")
print(f"Sisi Kanan (0.5 * |<[P_HV, P_45]>|) = {rhs_val:.6f}")
print(f"Apakah keadaan |R> merupakan keadaan ketidakpastian minimum? {np.isclose(lhs_val, rhs_val)}\n")

=== VERIFIKASI KETIDAKPASTIAN POLARISASI PADA KEADAAN |R> ===
P_R =
[[ 0.7071+0.j    ]
 [-0.    -0.7071j]]

Simpangan baku Delta P_HV        = 1.000000
Simpangan baku Delta P_45        = 1.000000
Sisi Kiri (Delta P_HV * Delta P_45) = 1.000000
Nilai Harap Komutator            = (-0-2j)
Sisi Kanan (0.5 * |<[P_HV, P_45]>|) = 1.000000
Apakah keadaan |R> merupakan keadaan ketidakpastian minimum? True



In [4]:
# Visualisasi Hubungan Ketidaktentuan untuk Keadaan Polarisasi Linier Sembarang
angles = np.linspace(0, 180, 200)
lhs_list = []
rhs_list = []
std_hv_list = []
std_45_list = []

for theta in angles:
    rad = deg(theta)
    # Keadaan polarisasi linier |theta> = cos(theta)|H> + sin(theta)|V>
    state = np.array([[np.cos(rad)], [np.sin(rad)]], dtype=complex)
    
    s_hv = std_dev(J_HV, state)
    s_45 = std_dev(J_45, state)
    
    std_hv_list.append(s_hv)
    std_45_list.append(s_45)
    lhs_list.append(s_hv * s_45)
    
    comm_exp = expectation_value(commutator(J_HV, J_45), state)
    rhs_list.append(0.5 * np.abs(comm_exp))

plt.figure(figsize=(10, 6))
plt.plot(angles, lhs_list, 'b-', lw=2.5, label=r'Hasil Kali Ketidakpastian $\Delta \mathcal{P}_{HV} \Delta \mathcal{P}_{45}$')
plt.plot(angles, rhs_list, 'r--', lw=2, label=r'Batas Bawah $\frac{1}{2}|\langle [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}] \rangle|$')
plt.plot(angles, std_hv_list, 'g:', alpha=0.7, label=r'$\Delta \mathcal{P}_{HV}$')
plt.plot(angles, std_45_list, 'm:', alpha=0.7, label=r'$\Delta \mathcal{P}_{45}$')

plt.title('Hubungan Ketidaktentuan pada Keadaan Polarisasi Linier $|\theta\rangle$', fontsize=14)
plt.xlabel(r'Sudut Polarisasi $\theta$ (derajat)', fontsize=12)
plt.ylabel('Nilai Ketidakpastian', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Interpretasi Visualisasi
Grafik di atas menunjukkan sifat ketidakpastian pengamatan $\hat{\mathcal{P}}_{HV}$ dan $\hat{\mathcal{P}}_{45}$ pada keadaan polarisasi linier $|\theta\rangle$:
- Pada sudut $\theta = 0^\circ$ ($|H\rangle$) atau $\theta = 90^\circ$ ($|V\rangle$), kepastian $\hat{\mathcal{P}}_{HV}$ maksimal ($\Delta \mathcal{P}_{HV} = 0$), sementara $\Delta \mathcal{P}_{45} = 1$.
- Pada sudut $\theta = 45^\circ$ atau $135^\circ$, kepastian $\hat{\mathcal{P}}_{45}$ maksimal ($\Delta \mathcal{P}_{45} = 0$), sementara $\Delta \mathcal{P}_{HV} = 1$.
- Karena pada keadaan linier murni nilai harap komutator $[\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}] = 2i\hat{\mathcal{P}}_C$ bernilai nol ($\langle \theta | \hat{\mathcal{P}}_C | \theta \rangle = 0$), batas bawah kurva merah bernilai nol di semua sudut linier. Ini menunjukkan bahwa kesamaan pada Persamaan (11) hanya dicapai penuh pada keadaan dengan komponen melingkar seperti $|R\rangle$ atau $|L\rangle$.

# Komplementaritas

Prinsip **Komplementaritas Bohr** menegaskan bahwa objek kuantum (seperti foton atau elektron) memiliki pasangan sifat komplementer yang saling eksklusif—sebagai contoh, **sifat gelombang (tak-terlokalisasi)** dan **sifat partikel (terlokalisasi)**. Kita dapat merancang eksperimen untuk mengamati salah satu sifat secara jelas, namun kita tidak dapat mengamati kedua sifat tersebut secara penuh pada saat yang sama.

### Analisis Eksperimen Interferometri Foton Tunggal (Mach-Zehnder)
Tinjau interferometer foton tunggal dengan dua jalur transmisi:
1. **Tanpa Detektor Jalur (*No Which-Way Information*)**:
   Jika tidak ada alat ukur internal di dalam interferometer, kita tidak memiliki informasi mengenai jalur mana yang dilewati foton. Foton melintasi kedua jalur sekaligus layaknya gelombang, sehingga ketika beda fase $\phi$ antar-jalur divariasikan, pola interferensi yang sempurna akan teramati pada probabilitas keluaran di detektor ($V = 1$).

2. **Dengan Detektor Jalur (*Full Which-Way Information*)**:
   Jika kita memasang perangkat penanda polarisasi nondestruktif di dalam interferometer (misalnya pelat setengah gelombang di satu lengan), foton di lengan atas mendapat polarisasi $|H\rangle$ dan di lengan bawah mendapat $|V\rangle$. Pengukuran ini memberikan informasi jalur secara sempurna (*which-way information*). Akibatnya, foton melintasi satu jalur layaknya partikel yang terlokalisasi, dan pola interferensi **hilang sepenuhnya** ($V = 0$).

3. **Informasi Parsial (*Partial Which-Way Information*)**:
   Jika penandaan jalur tidak sempurna (misalnya detektor hanya berhasil melokalisasi dengan probabilitas atau daya beda $D$), maka interferensi tidak hancur total melainkan berkurang visibilitasnya. Hubungan fundamental antara daya beda jalur (*distinguishability* $D$) dan visibilitas interferensi (*visibility* $V$) memenuhi **Ketidaksamaan Englert-Greenberger-Yasin**:

   $$
   D^2 + V^2 \le 1.
   $$

   Sebagai contoh, jika kita memiliki pengetahuan jalur sebesar 10% ($D = 0.1$), visibilitas maksimum pola interferensi dapat mencapai $\sqrt{1 - 0.1^2} \approx 99.5\%$. Jika kita mengetahui lintasannya pada 50% kesempatan ($D = 0.5$), visibilitas dapat berkurang menjadi $\sqrt{1 - 0.5^2} \approx 86.6\%$.

In [5]:
# Simulasi dan Visualisasi Komplementaritas: Duality Relation D^2 + V^2 <= 1
phases = np.linspace(0, 4 * np.pi, 300)

# Tiga kasus daya beda jalur (Distinguishability D)
D_values = [0.0, 0.6, 1.0]
labels = [
    'Murni Gelombang (D=0.0, V=1.0) - Interferensi Sempurna',
    'Parsial Komplementer (D=0.6, V=0.8) - Interferensi Teramati Sebagian',
    'Murni Partikel (D=1.0, V=0.0) - Interferensi Hilang Total'
]
colors = ['b', 'g', 'r']

plt.figure(figsize=(10, 6))

for D, label, col in zip(D_values, labels, colors):
    V = np.sqrt(1.0 - D**2)
    # Intensitas interferensi sebagai fungsi beda fase phi: I(phi) = I0 * (1 + V * cos(phi))
    intensity_out = 0.5 * (1.0 + V * np.cos(phases))
    plt.plot(phases, intensity_out, color=col, lw=2.5, label=label)

plt.title('Manifestasi Komplementaritas Bohr pada Keluaran Interferometer', fontsize=14)
plt.xlabel(r'Beda Fase Internal $\phi$ (radian)', fontsize=12)
plt.ylabel('Probabilitas / Intensitas Keluaran', fontsize=12)
plt.ylim(0, 1.05)
plt.legend(fontsize=10.5, loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Interpretasi Visualisasi Komplementaritas
Kurva biru merepresentasikan perilaku gelombang murni saat tidak ada informasi jalur yang bocor ($D=0$): probabilitas berosilasi penuh antara 0 dan 1. Kurva merah memperlihatkan perilaku partikel murni saat jalur diketahui pasti ($D=1$): probabilitas menjadi konstan di $0.5$ tanpa bergantung pada beda fase $\phi$. Kurva hijau menunjukkan keadaan transisi di mana pengamatan jalur bersifat parsial ($D=0.6$), mengilustrasikan kompromi kontinu antara sifat partikel dan gelombang.

# Soal-Jawab

## Soal 1: Sifat Anti-Hermitian Komutator

**Pertanyaan:** Suatu operator $\hat{O}$ dikatakan anti-Hermitian jika $\hat{O}^\dagger = -\hat{O}$. Jika $\hat{A}$ dan $\hat{B}$ adalah operator Hermitian ($\hat{A}^\dagger = \hat{A}$ dan $\hat{B}^\dagger = \hat{B}$), buktikan bahwa komutator $[\hat{A}, \hat{B}]$ bersifat anti-Hermitian.

### Jawaban Analitis & Pembuktian Matriks:
Sesuai definisi komutator:

$$
[\hat{A}, \hat{B}] = \hat{A}\hat{B} - \hat{B}\hat{A}.
$$

Kita ambil *adjoint* (konjugat transpos) dari kedua sisi dengan memanfaatkan sifat $(\hat{X} + \hat{Y})^\dagger = \hat{X}^\dagger + \hat{Y}^\dagger$ dan $(\hat{X}\hat{Y})^\dagger = \hat{Y}^\dagger \hat{X}^\dagger$:

$$
\begin{aligned}
[\hat{A}, \hat{B}]^\dagger &= (\hat{A}\hat{B} - \hat{B}\hat{A})^\dagger \\
&= (\hat{A}\hat{B})^\dagger - (\hat{B}\hat{A})^\dagger \\
&= \hat{B}^\dagger \hat{A}^\dagger - \hat{A}^\dagger \hat{B}^\dagger.
\end{aligned}
$$

Karena $\hat{A}$ dan $\hat{B}$ Hermitian, kita substitusikan $\hat{A}^\dagger = \hat{A}$ dan $\hat{B}^\dagger = \hat{B}$:

$$
\begin{aligned}
[\hat{A}, \hat{B}]^\dagger &= \hat{B}\hat{A} - \hat{A}\hat{B} \\
&= -(\hat{A}\hat{B} - \hat{B}\hat{A}) \\
&= -[\hat{A}, \hat{B}].
\end{aligned}
$$

Terbukti bahwa komutator dari dua operator Hermitian selalu bersifat **anti-Hermitian**.

In [6]:
# Verifikasi Numerik Soal 1 dengan Operator Polarisasi
print("=== VERIFIKASI SOAL 1: SIFAT ANTI-HERMITIAN KOMUTATOR ===")
comm_HV_45 = commutator(J_HV, J_45)
print_matrix("Matriks Komutator [J_HV, J_45]", comm_HV_45)

comm_adj = comm_HV_45.conj().T
print_matrix("Adjoint dari Komutator ([J_HV, J_45]^dagger)", comm_adj)
print_matrix("Negatif dari Komutator (-[J_HV, J_45])", -comm_HV_45)

print(f"Apakah [A, B]^dagger == -[A, B]? {np.allclose(comm_adj, -comm_HV_45)}\n")

=== VERIFIKASI SOAL 1: SIFAT ANTI-HERMITIAN KOMUTATOR ===
Matriks Komutator [J_HV, J_45] =
[[ 0.+0.j  2.+0.j]
 [-2.+0.j  0.+0.j]]

Adjoint dari Komutator ([J_HV, J_45]^dagger) =
[[ 0.+0.j -2.+0.j]
 [ 2.+0.j  0.+0.j]]

Negatif dari Komutator (-[J_HV, J_45]) =
[[ 0.+0.j -2.+0.j]
 [ 2.+0.j  0.+0.j]]

Apakah [A, B]^dagger == -[A, B]? True



## Soal 2: Representasi Matriks Operator Polarisasi $\hat{\mathcal{P}}_{45}$ dan $\hat{\mathcal{P}}_C$

**Pertanyaan:** Operator polarisasi $\hat{\mathcal{P}}_{HV}$ didefinisikan dalam basis spektralnya sebagai $\hat{\mathcal{P}}_{HV} = (+1)|H\rangle\langle H| + (-1)|V\rangle\langle V|$. Tentukanlah representasi matriks dalam basis $\{|H\rangle, |V\rangle\}$ untuk operator polarisasi $\hat{\mathcal{P}}_{45}$ dan $\hat{\mathcal{P}}_C$ yang bersesuaian dengan polarisasi linier diagonal $|\pm 45\rangle$ dan polarisasi melingkar $|L\rangle, |R\rangle$.

### 1. Representasi Matriks $\hat{\mathcal{P}}_{45}$:
Dengan analogi spektral, kita definisikan:

$$
\hat{\mathcal{P}}_{45} = (+1)|+45\rangle\langle +45| + (-1)|-45\rangle\langle -45|.
$$

Hubungan basis diagonal dengan basis $HV$ adalah:

$$
|+45\rangle = \frac{1}{\sqrt{2}}(|H\rangle + |V\rangle), \qquad |-45\rangle = \frac{1}{\sqrt{2}}(|H\rangle - |V\rangle).
$$

Operator proyeksi masing-masing keadaan adalah:

$$
|+45\rangle\langle +45| = \frac{1}{2}(|H\rangle + |V\rangle)(\langle H| + \langle V|) = \frac{1}{2}(|H\rangle\langle H| + |H\rangle\langle V| + |V\rangle\langle H| + |V\rangle\langle V|),
$$

$$
|-45\rangle\langle -45| = \frac{1}{2}(|H\rangle - |V\rangle)(\langle H| - \langle V|) = \frac{1}{2}(|H\rangle\langle H| - |H\rangle\langle V| - |V\rangle\langle H| + |V\rangle\langle V|).
$$

Mengurangkan keduanya memberikan:

$$
\hat{\mathcal{P}}_{45} = |+45\rangle\langle +45| - |-45\rangle\langle -45| = |H\rangle\langle V| + |V\rangle\langle H|.
$$

Dalam basis matriks kolom-baris $HV$ ($\{|H\rangle = \begin{pmatrix}1\\0\end{pmatrix}, |V\rangle = \begin{pmatrix}0\\1\end{pmatrix}\}$):

$$
\hat{\mathcal{P}}_{45} \doteq \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}_{HV}.
$$

### 2. Representasi Matriks $\hat{\mathcal{P}}_C$:
Untuk polarisasi melingkar, kita definisikan:

$$
\hat{\mathcal{P}}_C = (+1)|L\rangle\langle L| + (-1)|R\rangle\langle R|.
$$

Dengan relasi $|L\rangle = \frac{1}{\sqrt{2}}(|H\rangle + i|V\rangle)$ dan $|R\rangle = \frac{1}{\sqrt{2}}(|H\rangle - i|V\rangle)$:

$$
|L\rangle\langle L| = \frac{1}{2}(|H\rangle + i|V\rangle)(\langle H| - i\langle V|) = \frac{1}{2}(|H\rangle\langle H| - i|H\rangle\langle V| + i|V\rangle\langle H| + |V\rangle\langle V|),
$$

$$
|R\rangle\langle R| = \frac{1}{2}(|H\rangle - i|V\rangle)(\langle H| + i\langle V|) = \frac{1}{2}(|H\rangle\langle H| + i|H\rangle\langle V| - i|V\rangle\langle H| + |V\rangle\langle V|).
$$

Mengurangkan keduanya menghasilkan:

$$
\hat{\mathcal{P}}_C = |L\rangle\langle L| - |R\rangle\langle R| = -i|H\rangle\langle V| + i|V\rangle\langle H|.
$$

Dalam representasi matriks basis $HV$:

$$
\hat{\mathcal{P}}_C \doteq \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}_{HV}.

In [7]:
# Verifikasi Konstruksi Matriks P_45 dan P_C secara Spektral
print("=== VERIFIKASI SOAL 2: KONSTRUKSI MATRIKS DARI PROYEKSI SPEKTRAL ===")

# Proyeksi outer product untuk P_45
proj_plus_45  = P_plus_45 @ P_plus_45.conj().T
proj_minus_45 = P_minus_45 @ P_minus_45.conj().T
P_45_constructed = proj_plus_45 - proj_minus_45

# Proyeksi outer product untuk P_C
proj_L = P_L @ P_L.conj().T
proj_R = P_R @ P_R.conj().T
P_C_constructed = proj_L - proj_R

print_matrix("Matriks P_45 dari Proyeksi Spektral", P_45_constructed)
print_matrix("Matriks P_C dari Proyeksi Spektral", P_C_constructed)

print(f"Apakah P_45 hasil konstruksi sama dengan matriks analitis J_45? {np.allclose(P_45_constructed, J_45)}")
print(f"Apakah P_C hasil konstruksi sama dengan matriks analitis J_C? {np.allclose(P_C_constructed, J_C)}\n")

=== VERIFIKASI SOAL 2: KONSTRUKSI MATRIKS DARI PROYEKSI SPEKTRAL ===
Matriks P_45 dari Proyeksi Spektral =
[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]

Matriks P_C dari Proyeksi Spektral =
[[ 0.+0.j -0.-1.j]
 [ 0.+1.j  0.+0.j]]

Apakah P_45 hasil konstruksi sama dengan matriks analitis J_45? True
Apakah P_C hasil konstruksi sama dengan matriks analitis J_C? True



## Soal 3: Hubungan Komutasi Operator Polarisasi

**Pertanyaan:** Buktikan hubungan komutasi berikut menggunakan representasi matriks basis $HV$:
1. $[\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}] = 2i\hat{\mathcal{P}}_C$
2. $[\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_C] = -2i\hat{\mathcal{P}}_{45}$

### Pembuktian Langkah demi Langkah:
1. **Membuktikan $[\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}] = 2i\hat{\mathcal{P}}_C$**:
   Kita kalikan kedua matriks secara berurutan:

   $$
   \hat{\mathcal{P}}_{HV} \hat{\mathcal{P}}_{45} = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix},
   $$

   $$
   \hat{\mathcal{P}}_{45} \hat{\mathcal{P}}_{HV} = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}.
   $$

   Kurangkan keduanya:

   $$
   [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_{45}] = \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix} - \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} 0 & 2 \\ -2 & 0 \end{pmatrix}.
   $$

   Bandingkan dengan $2i\hat{\mathcal{P}}_C$:

   $$
   2i\hat{\mathcal{P}}_C = 2i \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix} = \begin{pmatrix} 0 & -2i^2 \\ 2i^2 & 0 \end{pmatrix} = \begin{pmatrix} 0 & 2 \\ -2 & 0 \end{pmatrix}. \qquad \text{(Terbukti!)}
   $$

2. **Membuktikan $[\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_C] = -2i\hat{\mathcal{P}}_{45}$**:
   Hitung hasil kali matriks:

   $$
   \hat{\mathcal{P}}_{HV} \hat{\mathcal{P}}_C = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix} = \begin{pmatrix} 0 & -i \\ -i & 0 \end{pmatrix},
   $$

   $$
   \hat{\mathcal{P}}_C \hat{\mathcal{P}}_{HV} = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} = \begin{pmatrix} 0 & i \\ i & 0 \end{pmatrix}.
   $$

   Kurangkan keduanya:

   $$
   [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_C] = \begin{pmatrix} 0 & -i \\ -i & 0 \end{pmatrix} - \begin{pmatrix} 0 & i \\ i & 0 \end{pmatrix} = \begin{pmatrix} 0 & -2i \\ -2i & 0 \end{pmatrix}.
   $$

   Bandingkan dengan $-2i\hat{\mathcal{P}}_{45}$:

   $$
   -2i\hat{\mathcal{P}}_{45} = -2i \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} 0 & -2i \\ -2i & 0 \end{pmatrix}. \qquad \text{(Terbukti!)}
   $$

In [8]:
# Verifikasi Numerik Soal 3
print("=== VERIFIKASI SOAL 3: HUBUNGAN KOMUTASI OPERATOR POLARISASI ===")
comm1 = commutator(J_HV, J_45)
target1 = 2j * J_C
print_matrix("Komutator [J_HV, J_45]", comm1)
print_matrix("2i * J_C", target1)
print(f"Apakah [J_HV, J_45] == 2i J_C? {np.allclose(comm1, target1)}\n")

comm2 = commutator(J_HV, J_C)
target2 = -2j * J_45
print_matrix("Komutator [J_HV, J_C]", comm2)
print_matrix("-2i * J_45", target2)
print(f"Apakah [J_HV, J_C] == -2i J_45? {np.allclose(comm2, target2)}\n")

=== VERIFIKASI SOAL 3: HUBUNGAN KOMUTASI OPERATOR POLARISASI ===
Komutator [J_HV, J_45] =
[[ 0.+0.j  2.+0.j]
 [-2.+0.j  0.+0.j]]

2i * J_C =
[[ 0.+0.j  2.+0.j]
 [-2.+0.j  0.+0.j]]

Apakah [J_HV, J_45] == 2i J_C? True

Komutator [J_HV, J_C] =
[[ 0.+0.j -0.-2.j]
 [ 0.-2.j  0.+0.j]]

-2i * J_45 =
[[ 0.+0.j -0.-2.j]
 [ 0.-2.j  0.+0.j]]

Apakah [J_HV, J_C] == -2i J_45? True



## Soal 4: Pembuktian Hubungan Ketidaktentuan pada Keadaan Polarisasi Eliptis

**Pertanyaan:** Buktikan bahwa pengamatan terkait operator $\hat{\mathcal{P}}_{HV}$ dan $\hat{\mathcal{P}}_C$ memenuhi hubungan ketidaktentuan untuk seberkas foton yang disiapkan dalam keadaan polarisasi eliptis:

$$
|e\rangle = \cos\theta |H\rangle + \sin\theta e^{i\phi} |V\rangle.
$$

### Bukti Lengkap:
Relasi ketidaktentuan yang harus diverifikasi adalah:

$$
\Delta \mathcal{P}_{HV} \Delta \mathcal{P}_C \ge \frac{1}{2} |\langle [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_C] \rangle|.
$$

Dalam basis $HV$, vektor kolom $|e\rangle$ dan vektor baris $\langle e|$ dituliskan sebagai:

$$
|e\rangle = \begin{pmatrix} \cos\theta \\ \sin\theta e^{i\phi} \end{pmatrix}, \qquad \langle e| = \begin{pmatrix} \cos\theta & \sin\theta e^{-i\phi} \end{pmatrix}.
$$

#### 1. Nilai Harap dan Simpangan Baku untuk $\hat{\mathcal{P}}_{HV}$:

$$
\hat{\mathcal{P}}_{HV} |e\rangle = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \begin{pmatrix} \cos\theta \\ \sin\theta e^{i\phi} \end{pmatrix} = \begin{pmatrix} \cos\theta \\ -\sin\theta e^{i\phi} \end{pmatrix}.
$$

Nilai harapnya:

$$
\langle \hat{\mathcal{P}}_{HV} \rangle = \langle e | \hat{\mathcal{P}}_{HV} | e \rangle = \begin{pmatrix} \cos\theta & \sin\theta e^{-i\phi} \end{pmatrix} \begin{pmatrix} \cos\theta \\ -\sin\theta e^{i\phi} \end{pmatrix} = \cos^2\theta - \sin^2\theta = \cos 2\theta.
$$

Karena $\hat{\mathcal{P}}_{HV}^2 = \hat{\mathbf{I}}$, maka $\langle \hat{\mathcal{P}}_{HV}^2 \rangle = 1$. Variansi dan simpangan bakunya:

$$
\Delta \mathcal{P}_{HV}^2 = 1 - \cos^2 2\theta = \sin^2 2\theta \implies \Delta \mathcal{P}_{HV} = |\sin 2\theta|.
$$

#### 2. Nilai Harap dan Simpangan Baku untuk $\hat{\mathcal{P}}_C$:

$$
\hat{\mathcal{P}}_C |e\rangle = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix} \begin{pmatrix} \cos\theta \\ \sin\theta e^{i\phi} \end{pmatrix} = \begin{pmatrix} -i\sin\theta e^{i\phi} \\ i\cos\theta \end{pmatrix}.
$$

Nilai harapnya:

$$
\begin{aligned}
\langle \hat{\mathcal{P}}_C \rangle &= \begin{pmatrix} \cos\theta & \sin\theta e^{-i\phi} \end{pmatrix} \begin{pmatrix} -i\sin\theta e^{i\phi} \\ i\cos\theta \end{pmatrix} \\
&= -i\cos\theta\sin\theta e^{i\phi} + i\sin\theta\cos\theta e^{-i\phi} \\
&= i\sin\theta\cos\theta (e^{-i\phi} - e^{i\phi}).
\end{aligned}
$$

Dengan identitas Euler $e^{-i\phi} - e^{i\phi} = -2i\sin\phi$, kita dapatkan:

$$
\langle \hat{\mathcal{P}}_C \rangle = i\sin\theta\cos\theta (-2i\sin\phi) = 2\sin\theta\cos\theta\sin\phi = \sin 2\theta \sin\phi.
$$

Karena $\hat{\mathcal{P}}_C^2 = \hat{\mathbf{I}}$, variansinya adalah:

$$
\Delta \mathcal{P}_C^2 = 1 - \sin^2 2\theta \sin^2\phi \implies \Delta \mathcal{P}_C = \sqrt{1 - \sin^2 2\theta \sin^2\phi}.
$$

#### 3. Nilai Harap Komutator dan Sisi Kanan:
Dari Soal 3, kita tahu $[\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_C] = -2i\hat{\mathcal{P}}_{45}$. Kita hitung $\langle \hat{\mathcal{P}}_{45} \rangle$ pada $|e\rangle$:

$$
\hat{\mathcal{P}}_{45} |e\rangle = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} \cos\theta \\ \sin\theta e^{i\phi} \end{pmatrix} = \begin{pmatrix} \sin\theta e^{i\phi} \\ \cos\theta \end{pmatrix}.
$$

$$
\begin{aligned}
\langle \hat{\mathcal{P}}_{45} \rangle &= \begin{pmatrix} \cos\theta & \sin\theta e^{-i\phi} \end{pmatrix} \begin{pmatrix} \sin\theta e^{i\phi} \\ \cos\theta \end{pmatrix} \\
&= \cos\theta\sin\theta e^{i\phi} + \sin\theta\cos\theta e^{-i\phi} \\
&= 2\sin\theta\cos\theta \cos\phi = \sin 2\theta \cos\phi.
\end{aligned}
$$

Maka modulus nilai harap komutator adalah:

$$
\frac{1}{2} |\langle [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_C] \rangle| = \frac{1}{2} |-2i \langle \hat{\mathcal{P}}_{45} \rangle| = |\sin 2\theta \cos\phi|.
$$

#### 4. Pembuktian Ketidaksamaan:
Kita bandingkan sisi kiri dan sisi kanan:

$$
|\sin 2\theta| \sqrt{1 - \sin^2 2\theta \sin^2\phi} \ge |\sin 2\theta \cos\phi|.
$$

Kuadratkan kedua sisi:

$$
\sin^2 2\theta (1 - \sin^2 2\theta \sin^2\phi) \ge \sin^2 2\theta \cos^2\phi.
$$

Gantikan $\cos^2\phi = 1 - \sin^2\phi$ di ruas kanan:

$$
\sin^2 2\theta - \sin^4 2\theta \sin^2\phi \ge \sin^2 2\theta - \sin^2 2\theta \sin^2\phi.
$$

Sederhanakan dengan memindahkan semua suku ke kiri:

$$
\sin^2 2\theta \sin^2\phi (1 - \sin^2 2\theta) \ge 0.
$$

Karena $1 - \sin^2 2\theta = \cos^2 2\theta \ge 0$, kuantitas di ruas kiri adalah perkalian kuadrat kuadrat riil yang selalu bernilai $\ge 0$. **Terbukti secara mutlak!**

In [9]:
# Verifikasi Numerik Soal 4 pada Grid Sembarang (theta, phi)
print("=== VERIFIKASI SOAL 4 PADA GRID SEMBARANG (THETA, PHI) ===")

thetas = np.linspace(0, np.pi, 100)
phis = np.linspace(0, 2 * np.pi, 50)
all_valid = True
min_diff = 1e9

for th in thetas:
    for ph in phis:
        state_e = np.array([[np.cos(th)], [np.sin(th) * np.exp(1j * ph)]], dtype=complex)
        
        s_hv = std_dev(J_HV, state_e)
        s_c  = std_dev(J_C, state_e)
        lhs  = s_hv * s_c
        
        comm_exp = expectation_value(commutator(J_HV, J_C), state_e)
        rhs  = 0.5 * np.abs(comm_exp)
        
        diff = lhs - rhs
        if diff < -1e-11:
            all_valid = False
        if diff < min_diff:
            min_diff = diff

print(f"Menguji {len(thetas)*len(phis)} kombinasi sudut eliptis (theta, phi)...")
print(f"Apakah seluruh {len(thetas)*len(phis)} sampel memenuhi hubungan ketidaktentuan? {all_valid}")
print(f"Selisih minimum (LHS - RHS) di seluruh grid = {min_diff:.10f}\n")

=== VERIFIKASI SOAL 4 PADA GRID SEMBARANG (THETA, PHI) ===
Menguji 5000 kombinasi sudut eliptis (theta, phi)...
Apakah seluruh 5000 sampel memenuhi hubungan ketidaktentuan? True
Selisih minimum (LHS - RHS) di seluruh grid = 0.0000000000



In [10]:
# Visualisasi 2D Contour: Selisih (LHS - RHS) pada Keadaan Eliptis |e>
THETA, PHI = np.meshgrid(np.linspace(0, 90, 100), np.linspace(0, 360, 100))
DIFF = np.zeros_like(THETA, dtype=float)

for i in range(THETA.shape[0]):
    for j in range(THETA.shape[1]):
        th_rad = deg(THETA[i, j])
        ph_rad = deg(PHI[i, j])
        
        # Sisi Kiri: |sin(2 theta)| * sqrt(1 - sin^2(2 theta) sin^2(phi))
        lhs = np.abs(np.sin(2 * th_rad)) * np.sqrt(np.maximum(0.0, 1.0 - (np.sin(2 * th_rad)**2) * (np.sin(ph_rad)**2)))
        # Sisi Kanan: |sin(2 theta) cos(phi)|
        rhs = np.abs(np.sin(2 * th_rad) * np.cos(ph_rad))
        
        DIFF[i, j] = lhs - rhs

plt.figure(figsize=(10, 7))
cp = plt.contourf(THETA, PHI, DIFF, levels=25, cmap='viridis')
cbar = plt.colorbar(cp)
cbar.set_label(r'Margin Ketidakpastian $\Delta \mathcal{P}_{HV} \Delta \mathcal{P}_C - \frac{1}{2}|\langle [\hat{\mathcal{P}}_{HV}, \hat{\mathcal{P}}_C] \rangle|$', fontsize=11)

plt.title(r'Peta Margin Ketidakpastian pada Keadaan Eliptis $|e\rangle$', fontsize=14)
plt.xlabel(r'Sudut Amplitudo $\theta$ (derajat)', fontsize=12)
plt.ylabel(r'Beda Fase Relatif $\phi$ (derajat)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# Ringkasan Pembelajaran

Dari penelusuran teoritis dan pembuktian komputasional dalam buku catatan ini, kita menyimpulkan beberapa poin inti:

1. **Batas Fundamental Ketidaktentuan Kuantum**: Hubungan $\Delta A \Delta B \ge \frac{1}{2}|\langle [\hat{A}, \hat{B}] \rangle|$ lahir langsung dari struktur aljabar ruang Hilbert kompleks dan ketidaksamaan Cauchy-Schwarz.
2. **Makna Filosofis *Indeterminacy***: Ketidaktentuan bukanlah ketidaksempurnaan pengukur, melainkan sifat intrinsik keadaan kuantum sebelum dilakukan pengamatan.
3. **Prinsip Komplementaritas Bohr**: Informasi jalur (*which-way info*) dan visibilitas interferensi bersaing secara kuantitatif dalam relasi dualitas $D^2 + V^2 \le 1$.
4. **Konsistensi Matriks**: Semua manipulasi abstrak dalam notasi Dirac sejalan seratus persen dengan operasi perkalian matriks $2 \times 2$, memberikan landasan komputasional yang kokoh bagi simulasi fisika kuantum.